<a href="https://colab.research.google.com/github/omarhany04/NLP-Projects/blob/main/nlp_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

importing libraries


In [ ]:
import string
from collections import Counter, defaultdict
import nltk
from datasets import load_dataset
from nltk.tokenize import word_tokenize


downloading necessary ntlk tokenizer models

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

loading wikitext dataset

In [ ]:
print("Loading WikiText-2 dataset...")
# fetching the raw Wikipedia corpus
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
# flattening the list of Wikipedia paragraphs into one continuous string for the training phase
train_text = "\n".join(dataset['train']['text'])

Loading WikiText-2 dataset...


Data pre-processing

In [ ]:
def preprocess_text(text):
    # converting all characters to lowercase
    text = text.lower()
    # removing all punctuation marks to focus on word sequences
    text = text.translate(str.maketrans('', '', string.punctuation))
    # spliting the long string into a Python list of individual word tokens
    tokens = word_tokenize(text)
    return tokens

# executing the preprocessing on the raw training text
print("Preprocessing tokens...")
tokens = preprocess_text(train_text)

Preprocessing tokens...


Constructing N-gram models & performing Laplace Smoothing

In [ ]:
class NGramModel:
    def __init__(self, n, tokens):
        # set N-gram order
        self.n = n
        # identify unique words in training data
        self.vocab = set(tokens)
        # total unique words (V) for smoothing
        self.vocab_size = len(self.vocab)
        # dictionary to store word frequencies after contexts
        self.ngram_counts = defaultdict(Counter)
        # frequency of each context appearing in text
        self.context_counts = Counter()
        # populate counts from tokens
        self._build_model(tokens)

    def _build_model(self, tokens):
        # sliding a window across the tokens to count occurrences of n-sized sequences
        for i in range(len(tokens) - self.n + 1):
            # the words the model looks at (n-1 words)
            context = tuple(tokens[i : i + self.n - 1])
            # the next word the model is trying to predict after context
            target = tokens[i + self.n - 1]

            self.ngram_counts[context][target] += 1
            self.context_counts[context] += 1

    def get_probability(self, word, context):

        #calculates the smoothed probability of a word given a context.
        #Formula: (Count of sequence + 1) / (Count of context + Vocabulary Size)
        context = tuple(context)
        count_w_context = self.ngram_counts[context][word]
        count_context = self.context_counts[context]

        # applying Laplace smoothing to avoid zero-probability errors
        probability = (count_w_context + 1) / (count_context + self.vocab_size)
        return probability

implementing different sized NGram models

In [ ]:
print("Building various N-gram models...")

# unigram (n=1): Context is empty tuple ()
unigram_model = NGramModel(n=1, tokens=tokens)

# bigram (n=2): Context is (word1,)
bigram_model  = NGramModel(n=2, tokens=tokens)

# trigram (n=3): Context is (word1, word2)
trigram_model = NGramModel(n=3, tokens=tokens)

# quadgram (n=4): Context is (word1, word2, word3)
quadgram_model = NGramModel(n=4, tokens=tokens)


Building various N-gram models...
